In [0]:
pip install openpyxl

In [0]:
%restart_python

In [0]:
import pandas as pd
import io

In [0]:
def standardize_col_name(col_name):
    return col_name.strip().replace(' ', '_').lower()

def rename_columns(df):
    old_columns = df.columns
    new_columns = [standardize_col_name(col) for col in old_columns]
    renamed_df = df.toDF(*new_columns)
    return renamed_df

In [0]:
metadata_content = spark.read.format("binaryFile").load("s3://de-mini-project-roshini/raw/meta_data.xlsx").collect()[0]["content"]
metadata_df = pd.read_excel(io.BytesIO(metadata_content))
data = spark.createDataFrame(metadata_df)

for d in data.collect():
    file_content = spark.read.format("binaryFile").load(f"s3://de-mini-project-roshini/raw/{d.file_name}.{d.extension}").collect()[0]["content"]
    pandas_df = pd.read_excel(io.BytesIO(file_content))
    for col in pandas_df.columns:
        if pandas_df[col].dtype == 'object':
            pandas_df[col] = pandas_df[col].astype(str)
    temp = spark.createDataFrame(pandas_df)
    temp = rename_columns(temp)
    temp.write.mode("overwrite").format("delta").saveAsTable(f"globaldistributor.{d.schema_name}.{d.table_name}")